In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Telecom Customer Usage and Billing Analytics") \
    .getOrCreate()

In [3]:
customers_df=spark.read.csv(
    "customer.csv",
    header=True,
    inferSchema=True
)

In [4]:
usage_df=spark.read.csv(
    "usage.csv",
    header=True,
    inferSchema=True
)

In [5]:
payments_df=spark.read.csv(
    "payments.csv",
    header=True,
    inferSchema=True
)

In [6]:
plans_df = spark.read.option(
    "multiline", "true"
).json("plans.json")

In [7]:
customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)



In [8]:
usage_df.printSchema()

root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)



In [9]:
payments_df.printSchema()

root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [10]:
plans_df.printSchema()

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)



In [11]:
customers_df.count()

12

In [12]:
usage_df.count()

15

In [13]:
payments_df.count()

15

In [14]:
plans_df.count()

4

In [15]:
customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [16]:
usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [17]:
payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [18]:
from pyspark.core.rdd import T
plans_df.show(truncate=False)

+-------------+---------------------------+-----------+-------+------------+
|data_limit_gb|features                   |monthly_fee|plan_id|plan_name   |
+-------------+---------------------------+-----------+-------+------------+
|50           |{false, National, true}    |499        |P101   |Smart Basic |
|75           |{true, National, true}     |799        |P102   |Smart Plus  |
|25           |{false, NULL, false}       |299        |P103   |Budget Saver|
|100          |{true, International, true}|1199       |P104   |Premium Max |
+-------------+---------------------------+-----------+-------+------------+



In [19]:
customers_df.write.mode("overwrite") \
.parquet("bronze/customers")

In [20]:
usage_df.write.mode("overwrite") \
.parquet("bronze/usage")

In [21]:
payments_df.write.mode("overwrite") \
.parquet("bronze/payments")

In [22]:
plans_df.write.mode("overwrite") \
.parquet("bronze/plans")

In [23]:
customers_df.filter(
    col("plan_id").isNull()
).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [24]:
usage_df.filter(
    col("data_used_gb").isNull()
).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [25]:
payments_df.filter(
    col("amount_paid").isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [26]:
payments_df.filter(
    col("payment_mode").isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [27]:
usage_clean = usage_df.fillna({
    "data_used_gb":0
})
usage_clean.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [28]:
payments_clean = payments_df.fillna({
    "amount_paid":0
})
payments_clean.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [29]:
payments_clean = payments_clean.fillna({
    "payment_mode":"Not Provided"
})
payments_clean.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [30]:
customers_clean = customers_df.fillna({
    "plan_id":"UNKNOWN"
})
customers_clean.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [31]:
customers_clean = customers_clean.withColumn(
    "data_quality_status",
    when(col("plan_id")=="UNKNOWN","Missing Plan")
    .otherwise("Valid")
)
customers_clean.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|              Valid|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|              Valid|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|              Valid|
|        108|   Meer

In [32]:
usage_clean = usage_clean.withColumn(
    "data_quality_status",
    when(col("data_used_gb")==0,"Missing Data Usage")
    .otherwise("Valid")
)
usage_clean.show()

+--------+-----------+-------------------+------------+------------+---------+-------------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-------------------+------------+------------+---------+-------------------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|              Valid|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|              Valid|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|              Valid|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|              Valid|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|              Valid|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|              Valid|
|    1008|

In [33]:
payments_clean = payments_clean.withColumn(
    "data_quality_status",
    when(col("amount_paid")==0,"Missing Amount")
    .when(col("payment_mode")=="Not Provided","Missing Mode")
    .otherwise("Valid")
)
payments_clean.show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|              Valid|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Valid|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|              Valid|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|              Valid|
|      5007|        107|2026-01-01 00:00:00|        299

In [34]:
customers_clean.write.mode("overwrite") \
.parquet("silver/customers")

usage_clean.write.mode("overwrite") \
.parquet("silver/usage")

payments_clean.write.mode("overwrite") \
.parquet("silver/payments")

In [35]:
plans_df.printSchema()

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)



In [36]:
plans_flat = plans_df.select(
    col("plan_id"),
    col("plan_name"),
    col("monthly_fee"),
    col("data_limit_gb"),
    col("features.unlimited_calls"),
    col("features.ott_included"),
    col("features.roaming")
)

In [37]:
plans_flat.show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |NULL         |
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [38]:
plans_flat.select(
    "plan_id",
    "unlimited_calls"
).show()

+-------+---------------+
|plan_id|unlimited_calls|
+-------+---------------+
|   P101|           true|
|   P102|           true|
|   P103|          false|
|   P104|           true|
+-------+---------------+



In [39]:
plans_flat.select(
    "plan_id",
    "ott_included"
).show()

+-------+------------+
|plan_id|ott_included|
+-------+------------+
|   P101|       false|
|   P102|        true|
|   P103|       false|
|   P104|        true|
+-------+------------+



In [40]:
plans_flat.select(
    "plan_id",
    "roaming"
).show()

+-------+-------------+
|plan_id|      roaming|
+-------+-------------+
|   P101|     National|
|   P102|     National|
|   P103|         NULL|
|   P104|International|
+-------+-------------+



In [41]:
plans_flat = plans_flat.fillna(
    {"roaming":"Not Available"}
)

In [42]:
plans_flat.show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |Not Available|
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [43]:
plans_flat.write.mode("overwrite") \
.parquet("silver/plans")

In [44]:
plans_flat = spark.read.parquet("silver/plans")
plans_flat.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|Not Available|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [45]:
customer_plan_df = customers_clean.join(
    plans_flat,
    "plan_id",
    "left"
    )

customer_plan_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|              Valid| Smart Basic|        499|           50|           true|       false|     National|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|              Valid|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|Inactive|              Valid|Bud

In [46]:
customer_usage_df = customers_clean.join(
    usage_clean, "customer_id", "left")

customer_usage_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1012|2026-02-01 00:00:00|          50|        1000|      130|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1001|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|        

In [47]:
customer_payment_df = customers_clean.join(
    payments_clean, "customer_id", "left")

customer_payment_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5012|2026-02-01 00:00:00|        499|        Card|       Success|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5001|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Fe

In [48]:
complete_df = customers_clean \
    .join(plans_flat, "plan_id", "left") \
    .join(usage_clean, "customer_id", "left") \
    .join(payments_clean, "customer_id", "left")

complete_df.show(truncate=False)

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|city     |state      |age|gender|status  |data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------

In [49]:
invalid_plan_df = customers_clean.join(
    plans_flat, "plan_id", "left")

invalid_plan_df.filter(
    col("plan_name").isNull()
    ).show()

+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+------------+-------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|data_quality_status|plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+------------+-------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|              Valid|     NULL|       NULL|         NULL|           NULL|        NULL|   NULL|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|       Missing Plan|     NULL|       NULL|         NULL|           NULL|        NULL|   NULL|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+---------+-----------+-------------+---------------+--------

In [50]:
invalid_usage_df = usage_clean.join(
    customers_clean, "customer_id", "leftanti")

invalid_usage_df.show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|              Valid|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [51]:
invalid_payment_df = payments_clean.join(
    customers_clean, "customer_id", "leftanti")

invalid_payment_df.show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [52]:
complete_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------

In [53]:
complete_df = complete_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Low User")
)

complete_df.select(
    "customer_id","data_used_gb","usage_category"
    ).show()

+-----------+------------+--------------+
|customer_id|data_used_gb|usage_category|
+-----------+------------+--------------+
|        101|          50|   Medium User|
|        101|          50|   Medium User|
|        101|          45|   Medium User|
|        101|          45|   Medium User|
|        102|          34|   Medium User|
|        102|          34|   Medium User|
|        102|          30|   Medium User|
|        102|          30|   Medium User|
|        103|          12|      Low User|
|        104|          58|   Medium User|
|        104|          58|   Medium User|
|        104|          55|   Medium User|
|        104|          55|   Medium User|
|        105|           0|      Low User|
|        105|           0|      Low User|
|        105|          75|    Heavy User|
|        105|          75|    Heavy User|
|        106|          28|      Low User|
|        107|          10|      Low User|
|        108|          80|    Heavy User|
+-----------+------------+--------

In [54]:
complete_df = complete_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000, "High Payment")
    .when(col("amount_paid") >= 500, "Medium Payment")
    .otherwise("Low Payment")
)

complete_df.select(
    "customer_id","amount_paid","payment_category"
    ).show()

+-----------+-----------+----------------+
|customer_id|amount_paid|payment_category|
+-----------+-----------+----------------+
|        101|        499|     Low Payment|
|        101|        499|     Low Payment|
|        101|        499|     Low Payment|
|        101|        499|     Low Payment|
|        102|        799|  Medium Payment|
|        102|        799|  Medium Payment|
|        102|        799|  Medium Payment|
|        102|        799|  Medium Payment|
|        103|        299|     Low Payment|
|        104|        499|     Low Payment|
|        104|        499|     Low Payment|
|        104|        499|     Low Payment|
|        104|        499|     Low Payment|
|        105|       1199|    High Payment|
|        105|       1199|    High Payment|
|        105|       1199|    High Payment|
|        105|       1199|    High Payment|
|        106|        799|  Medium Payment|
|        107|        299|     Low Payment|
|        108|       1199|    High Payment|
+----------

In [55]:
complete_df = complete_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(col("data_used_gb") < 15, "Medium Risk")
    .otherwise("Low Risk")
)

complete_df.select(
    "customer_id",
    "status",
    "payment_status",
    "data_used_gb",
    "churn_risk"
).show()

+-----------+--------+--------------+------------+-----------+
|customer_id|  status|payment_status|data_used_gb| churn_risk|
+-----------+--------+--------------+------------+-----------+
|        101|  Active|       Success|          50|   Low Risk|
|        101|  Active|       Success|          50|   Low Risk|
|        101|  Active|       Success|          45|   Low Risk|
|        101|  Active|       Success|          45|   Low Risk|
|        102|  Active|       Success|          34|   Low Risk|
|        102|  Active|       Success|          34|   Low Risk|
|        102|  Active|       Success|          30|   Low Risk|
|        102|  Active|       Success|          30|   Low Risk|
|        103|Inactive|        Failed|          12|  High Risk|
|        104|  Active|       Success|          58|   Low Risk|
|        104|  Active|       Success|          58|   Low Risk|
|        104|  Active|       Success|          55|   Low Risk|
|        104|  Active|       Success|          55|   Lo

In [56]:
complete_df = complete_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)

complete_df.select(
    "customer_id",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
).show()

+-----------+------------+-------------+-------------+
|customer_id|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+------------+-------------+-------------+
|        101|          50|           50|            0|
|        101|          50|           50|            0|
|        101|          45|           50|           -5|
|        101|          45|           50|           -5|
|        102|          34|           75|          -41|
|        102|          34|           75|          -41|
|        102|          30|           75|          -45|
|        102|          30|           75|          -45|
|        103|          12|           25|          -13|
|        104|          58|           50|            8|
|        104|          58|           50|            8|
|        104|          55|           50|            5|
|        104|          55|           50|            5|
|        105|           0|          100|         -100|
|        105|           0|          100|         -100|
|        1

In [57]:
complete_df = complete_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes")
    .otherwise("No")
)

complete_df.select(
    "customer_id",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb",
    "over_usage_flag"
).show()

+-----------+------------+-------------+-------------+---------------+
|customer_id|data_used_gb|data_limit_gb|over_usage_gb|over_usage_flag|
+-----------+------------+-------------+-------------+---------------+
|        101|          50|           50|            0|             No|
|        101|          50|           50|            0|             No|
|        101|          45|           50|           -5|             No|
|        101|          45|           50|           -5|             No|
|        102|          34|           75|          -41|             No|
|        102|          34|           75|          -41|             No|
|        102|          30|           75|          -45|             No|
|        102|          30|           75|          -45|             No|
|        103|          12|           25|          -13|             No|
|        104|          58|           50|            8|            Yes|
|        104|          58|           50|            8|            Yes|
|     

In [58]:
complete_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    5|
|    Kochi|    1|
|  Chennai|    4|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    5|
|Hyderabad|    6|
+---------+-----+



In [59]:
complete_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    5|
|     Kerala|    1|
| Tamil Nadu|    4|
|      Delhi|    5|
|  Telangana|    6|
|Maharashtra|    3|
+-----------+-----+



In [60]:
complete_df.groupBy("plan_name").count().show()

+------------+-----+
|   plan_name|count|
+------------+-----+
|        NULL|    2|
| Smart Basic|    9|
|Budget Saver|    2|
| Premium Max|    5|
|  Smart Plus|    6|
+------------+-----+



In [61]:
complete_df.groupBy("usage_category").count().show()

+--------------+-----+
|usage_category|count|
+--------------+-----+
|   Medium User|   14|
|    Heavy User|    3|
|      Low User|    7|
+--------------+-----+



In [62]:
complete_df.groupBy("churn_risk").count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|   Low Risk|   19|
|Medium Risk|    1|
|  High Risk|    4|
+-----------+-----+



In [63]:
complete_df.groupBy("plan_name") \
    .sum("data_used_gb") \
    .show()

+------------+-----------------+
|   plan_name|sum(data_used_gb)|
+------------+-----------------+
|        NULL|             NULL|
| Smart Basic|              464|
|Budget Saver|               22|
| Premium Max|              230|
|  Smart Plus|              188|
+------------+-----------------+



In [64]:
complete_df.groupBy("plan_name") \
    .agg(avg("data_used_gb").alias("average_data_usage")) \
    .show()

+------------+------------------+
|   plan_name|average_data_usage|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic| 51.55555555555556|
|Budget Saver|              11.0|
| Premium Max|              46.0|
|  Smart Plus|31.333333333333332|
+------------+------------------+



In [65]:
complete_df.groupBy("city") \
    .agg(sum("call_minutes").alias("total_call_minutes")) \
    .show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              3450|
|    Kochi|              1600|
|  Chennai|              4600|
|   Mumbai|               250|
|     Pune|               500|
|    Delhi|              6600|
|Hyderabad|              4000|
+---------+------------------+



In [66]:
complete_df.groupBy("state") \
    .agg(sum("sms_count").alias("total_sms")) \
    .show()

+-----------+---------+
|      state|total_sms|
+-----------+---------+
|  Karnataka|      430|
|     Kerala|      250|
| Tamil Nadu|      620|
|      Delhi|      910|
|  Telangana|      520|
|Maharashtra|      100|
+-----------+---------+



In [67]:
complete_df.filter(col("payment_status")=="Success") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .show()

+-------------+
|total_revenue|
+-------------+
|        12882|
+-------------+



In [68]:
complete_df.groupBy("city") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         3695|
|    Kochi|         1199|
|  Chennai|         1996|
|   Mumbai|          299|
|     Pune|          799|
|    Delhi|         5595|
|Hyderabad|         2295|
+---------+-------------+



In [69]:
complete_df.groupBy("plan_name") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [70]:
complete_df.groupBy("payment_mode") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        NULL|         NULL|
|        Card|         6193|
|        Cash|          598|
|Not Provided|         2398|
|         UPI|         6689|
+------------+-------------+



In [71]:
complete_df.groupBy("plan_name") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .show(1)

+-----------+-------------+
|  plan_name|total_revenue|
+-----------+-------------+
|Premium Max|         5995|
+-----------+-------------+
only showing top 1 row


In [72]:
complete_df.groupBy("city") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .show(1)

+-----+-------------+
| city|total_revenue|
+-----+-------------+
|Delhi|         5595|
+-----+-------------+
only showing top 1 row


In [73]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [74]:
window_spec = Window.orderBy(col("data_used_gb").desc())

rank_usage_df = complete_df.withColumn(
    "usage_rank",
    rank().over(window_spec)
)

rank_usage_df.select(
    "customer_id",
    "customer_name",
    "data_used_gb",
    "usage_rank"
).show()

+-----------+-------------+------------+----------+
|customer_id|customer_name|data_used_gb|usage_rank|
+-----------+-------------+------------+----------+
|        108|   Meera Nair|          80|         1|
|        105|   Farhan Ali|          75|         2|
|        105|   Farhan Ali|          75|         2|
|        104|  Sneha Patel|          58|         4|
|        104|  Sneha Patel|          58|         4|
|        104|  Sneha Patel|          55|         6|
|        104|  Sneha Patel|          55|         6|
|        101| Rahul Sharma|          50|         8|
|        101| Rahul Sharma|          50|         8|
|        109|    Kiran Rao|          48|        10|
|        101| Rahul Sharma|          45|        11|
|        101| Rahul Sharma|          45|        11|
|        102|  Priya Reddy|          34|        13|
|        102|  Priya Reddy|          34|        13|
|        110|  Nisha Reddy|          32|        15|
|        102|  Priya Reddy|          30|        16|
|        102

In [75]:
window_spec = Window.orderBy(col("amount_paid").desc())

rank_payment_df = complete_df.withColumn(
    "payment_rank",
    rank().over(window_spec)
)

rank_payment_df.select(
    "customer_id",
    "customer_name",
    "amount_paid",
    "payment_rank"
).show()

+-----------+-------------+-----------+------------+
|customer_id|customer_name|amount_paid|payment_rank|
+-----------+-------------+-----------+------------+
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        108|   Meera Nair|       1199|           1|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        106|   Neha Singh|        799|           6|
|        110|  Nisha Reddy|        799|           6|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        104|  Sneha Patel|        499|       

In [76]:
top3_users = complete_df.withColumn(
    "rank",
    rank().over(window_spec)
)

top3_users.filter(col("rank") <= 3).show()

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+----+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-

In [77]:
top3_revenue = complete_df.withColumn(
    "rank",
    rank().over(window_spec)
)

top3_revenue.filter(col("rank") <= 3).show()

+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+----+
|customer_id|plan_id|customer_name| city| state|age|gender|status|data_quality_status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+-----+------+---+------+------+-------------------+-----------+-----------+-------------+---------------+------------+-

In [78]:
window_spec = Window.partitionBy("city").orderBy(col("amount_paid").desc())

top_city_customer = complete_df.withColumn(
    "rank",
    rank().over(window_spec)
)

top_city_customer.filter(col("rank") == 1).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+--------

In [79]:
top_plan_customer = complete_df.withColumn(
    "rank",
    rank().over(window_spec)
)

top_plan_customer.filter(col("rank") == 1).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+--------------+----------------+-----------+-------------+---------------+----+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+--------

In [80]:
window_spec = Window.orderBy("bill_month")

running_revenue_df = complete_df.withColumn(
    "running_total_revenue",
    sum("amount_paid").over(window_spec)
)

running_revenue_df.select(
    "bill_month",
    "amount_paid",
    "running_total_revenue"
).show()

+-------------------+-----------+---------------------+
|         bill_month|amount_paid|running_total_revenue|
+-------------------+-----------+---------------------+
|               NULL|       NULL|                 NULL|
|2026-01-01 00:00:00|        499|                 9886|
|2026-01-01 00:00:00|        499|                 9886|
|2026-01-01 00:00:00|        799|                 9886|
|2026-01-01 00:00:00|        799|                 9886|
|2026-01-01 00:00:00|        299|                 9886|
|2026-01-01 00:00:00|        499|                 9886|
|2026-01-01 00:00:00|        499|                 9886|
|2026-01-01 00:00:00|       1199|                 9886|
|2026-01-01 00:00:00|       1199|                 9886|
|2026-01-01 00:00:00|        799|                 9886|
|2026-01-01 00:00:00|        299|                 9886|
|2026-01-01 00:00:00|       1199|                 9886|
|2026-01-01 00:00:00|        499|                 9886|
|2026-01-01 00:00:00|        799|               

In [81]:
window_spec = Window.partitionBy("customer_id").orderBy("usage_month")

lag_df = complete_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb",1).over(window_spec)
)

lag_df.select(
    "customer_id",
    "usage_month",
    "data_used_gb",
    "previous_month_usage"
).show()

+-----------+-------------------+------------+--------------------+
|customer_id|        usage_month|data_used_gb|previous_month_usage|
+-----------+-------------------+------------+--------------------+
|        101|2026-01-01 00:00:00|          45|                NULL|
|        101|2026-01-01 00:00:00|          45|                  45|
|        101|2026-02-01 00:00:00|          50|                  45|
|        101|2026-02-01 00:00:00|          50|                  50|
|        102|2026-01-01 00:00:00|          30|                NULL|
|        102|2026-01-01 00:00:00|          30|                  30|
|        102|2026-02-01 00:00:00|          34|                  30|
|        102|2026-02-01 00:00:00|          34|                  34|
|        103|2026-01-01 00:00:00|          12|                NULL|
|        104|2026-01-01 00:00:00|          55|                NULL|
|        104|2026-01-01 00:00:00|          55|                  55|
|        104|2026-02-01 00:00:00|          58|  

In [82]:
window_spec = Window.partitionBy("customer_id").orderBy("usage_month")

lead_df = complete_df.withColumn(
    "next_month_usage",
    lead("data_used_gb",1).over(window_spec)
)

lead_df.select(
    "customer_id",
    "usage_month",
    "data_used_gb",
    "next_month_usage"
).show()

+-----------+-------------------+------------+----------------+
|customer_id|        usage_month|data_used_gb|next_month_usage|
+-----------+-------------------+------------+----------------+
|        101|2026-01-01 00:00:00|          45|              45|
|        101|2026-01-01 00:00:00|          45|              50|
|        101|2026-02-01 00:00:00|          50|              50|
|        101|2026-02-01 00:00:00|          50|            NULL|
|        102|2026-01-01 00:00:00|          30|              30|
|        102|2026-01-01 00:00:00|          30|              34|
|        102|2026-02-01 00:00:00|          34|              34|
|        102|2026-02-01 00:00:00|          34|            NULL|
|        103|2026-01-01 00:00:00|          12|            NULL|
|        104|2026-01-01 00:00:00|          55|              55|
|        104|2026-01-01 00:00:00|          55|              58|
|        104|2026-02-01 00:00:00|          58|              58|
|        104|2026-02-01 00:00:00|       

In [83]:
window_spec = Window.partitionBy("customer_id").orderBy("usage_month")

increase_df = complete_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb",1).over(window_spec)
)

increase_df.filter(
    col("data_used_gb") > col("previous_month_usage")
).select(
    "customer_id",
    "customer_name",
    "usage_month",
    "data_used_gb",
    "previous_month_usage"
).show()

+-----------+-------------+-------------------+------------+--------------------+
|customer_id|customer_name|        usage_month|data_used_gb|previous_month_usage|
+-----------+-------------+-------------------+------------+--------------------+
|        101| Rahul Sharma|2026-02-01 00:00:00|          50|                  45|
|        102|  Priya Reddy|2026-02-01 00:00:00|          34|                  30|
|        104|  Sneha Patel|2026-02-01 00:00:00|          58|                  55|
+-----------+-------------+-------------------+------------+--------------------+



In [84]:
customers_clean.createOrReplaceTempView("customers")

usage_clean.createOrReplaceTempView("usage")

payments_clean.createOrReplaceTempView("payments")

plans_flat.createOrReplaceTempView("plans")

complete_df.createOrReplaceTempView("customer_usage_billing")

In [85]:
spark.sql("""
select* from customers
where status='Active'
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|              Valid|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|              Valid|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|              Valid|
|        110|  Nisha Reddy|    Delhi|   

In [86]:
spark.sql("""
select city,
count(*) as total_customers
from customers
group by city
""").show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              2|
|    Kochi|              1|
|  Chennai|              1|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              2|
|Hyderabad|              3|
+---------+---------------+



In [87]:
spark.sql("""
select plan_name,
sum(amount_paid) as total_revenue
from customer_usage_billing
group by plan_name
""").show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [88]:
spark.sql("""
select customer_id,
customer_name,
data_used_gb
from customer_usage_billing
where data_used_gb >= 70
""").show()

+-----------+-------------+------------+
|customer_id|customer_name|data_used_gb|
+-----------+-------------+------------+
|        105|   Farhan Ali|          75|
|        105|   Farhan Ali|          75|
|        108|   Meera Nair|          80|
+-----------+-------------+------------+



In [89]:
spark.sql("""
select customer_id,
customer_name,
churn_risk
from customer_usage_billing
where churn_risk='High Risk'
""").show()

+-----------+-------------+----------+
|customer_id|customer_name|churn_risk|
+-----------+-------------+----------+
|        103|   Amit Kumar| High Risk|
|        105|   Farhan Ali| High Risk|
|        105|   Farhan Ali| High Risk|
|        107|  Arjun Verma| High Risk|
+-----------+-------------+----------+



In [90]:
spark.sql("""
select customer_id,
customer_name,
plan_id
from customers
where plan_id='UNKNOWN'
""").show()

+-----------+-------------+-------+
|customer_id|customer_name|plan_id|
+-----------+-------------+-------+
|        112|  Ayesha Khan|UNKNOWN|
+-----------+-------------+-------+



In [91]:
spark.sql("""
select customer_id,
payment_status,
amount_paid
from customer_usage_billing
where payment_status in ('Failed','Pending')
""").show()

+-----------+--------------+-----------+
|customer_id|payment_status|amount_paid|
+-----------+--------------+-----------+
|        103|        Failed|        299|
|        105|       Pending|       1199|
|        105|       Pending|       1199|
|        107|       Pending|        299|
+-----------+--------------+-----------+



In [92]:
spark.sql("""
select customer_id,
customer_name,
data_used_gb
from customer_usage_billing
order by data_used_gb desc
limit 5
""").show()

+-----------+-------------+------------+
|customer_id|customer_name|data_used_gb|
+-----------+-------------+------------+
|        108|   Meera Nair|          80|
|        105|   Farhan Ali|          75|
|        105|   Farhan Ali|          75|
|        104|  Sneha Patel|          58|
|        104|  Sneha Patel|          58|
+-----------+-------------+------------+



In [93]:
spark.sql("""
select payment_mode,
sum(amount_paid) as total_revenue
from customer_usage_billing
group by payment_mode
""").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|        NULL|         NULL|
|        Card|         6193|
|        Cash|          598|
|Not Provided|         2398|
|         UPI|         6689|
+------------+-------------+



In [94]:
customers_clean = customers_clean.withColumnRenamed(
    "data_quality_status", "customer_quality_status")

usage_clean = usage_clean.withColumnRenamed(
    "data_quality_status", "usage_quality_status")

payments_clean = payments_clean.withColumnRenamed(
    "data_quality_status", "payment_quality_status")

In [95]:
complete_df = customers_clean \
.join(plans_flat, "plan_id", "left") \
.join(usage_clean, "customer_id", "left") \
.join(payments_clean, "customer_id", "left")

In [96]:
complete_df.write.mode("overwrite") \
.parquet("gold/customer_usage_summary")

In [97]:
complete_df.write.mode("overwrite") \
.partitionBy("usage_month") \
.parquet("gold/customer_usage_summary")

In [98]:
%%writefile usage_incremental.csv

usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,55,1100,140
1017,102,2026-03,40,700,95
1018,104,2026-03,62,1250,170
1019,105,2026-03,85,1550,220
1020,108,2026-03,90,1700,270

Overwriting usage_incremental.csv


In [107]:
usage_incremental_df = spark.read.csv(
    "usage_incremental.csv",
    header=True,
    inferSchema=True
)

usage_incremental_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1016|        101|2026-03-01 00:00:00|          55|        1100|      140|
|    1017|        102|2026-03-01 00:00:00|          40|         700|       95|
|    1018|        104|2026-03-01 00:00:00|          62|        1250|      170|
|    1019|        105|2026-03-01 00:00:00|          85|        1550|      220|
|    1020|        108|2026-03-01 00:00:00|          90|        1700|      270|
+--------+-----------+-------------------+------------+------------+---------+



In [109]:
from pyspark.sql.functions import lit

usage_incremental_df = usage_incremental_df.withColumn(
    "data_quality_status",
    lit("Valid")
)

silver_usage_df = usage_clean

updated_usage_df = silver_usage_df.unionByName(
    usage_incremental_df,
    allowMissingColumns=True
)

updated_usage_df.write.mode("overwrite") \
.parquet("silver/usage_updated")

In [110]:
updated_complete_df = customers_clean \
.join(plans_flat, "plan_id", "left") \
.join(updated_usage_df, "customer_id", "left") \
.join(payments_clean, "customer_id", "left")

In [111]:
updated_complete_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+--------------------+-------------------+----------+-------------------+-----------+------------+--------------+----------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|customer_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|usage_quality_status|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|payment_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-----------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+-------

In [112]:
customers_clean = customers_clean.withColumnRenamed(
    "data_quality_status",
    "customer_quality_status"
)

updated_usage_df = updated_usage_df.withColumnRenamed(
    "data_quality_status",
    "usage_quality_status"
)

payments_clean = payments_clean.withColumnRenamed(
    "data_quality_status",
    "payment_quality_status"
)

In [115]:
gold_df = customers_clean \
.join(plans_flat, "plan_id", "left") \
.join(updated_usage_df, "customer_id", "left") \
.join(payments_clean, "customer_id", "left")

In [117]:
print(gold_df.columns)

['customer_id', 'plan_id', 'customer_name', 'city', 'state', 'age', 'gender', 'status', 'customer_quality_status', 'plan_name', 'monthly_fee', 'data_limit_gb', 'unlimited_calls', 'ott_included', 'roaming', 'usage_id', 'usage_month', 'data_used_gb', 'call_minutes', 'sms_count', 'usage_quality_status', 'usage_quality_status', 'payment_id', 'bill_month', 'amount_paid', 'payment_mode', 'payment_status', 'payment_quality_status']


In [118]:
gold_df = customers_clean \
    .join(plans_flat, "plan_id", "left") \
    .join(updated_usage_df, "customer_id", "left") \
    .join(payments_clean, "customer_id", "left")

In [119]:
gold_df = gold_df.toDF(*[
    f"{c}_{i}" if gold_df.columns.count(c) > 1 and i > 0 else c
    for i, c in enumerate(gold_df.columns)
])

In [120]:
print(gold_df.columns)

['customer_id', 'plan_id', 'customer_name', 'city', 'state', 'age', 'gender', 'status', 'customer_quality_status', 'plan_name', 'monthly_fee', 'data_limit_gb', 'unlimited_calls', 'ott_included', 'roaming', 'usage_id', 'usage_month', 'data_used_gb', 'call_minutes', 'sms_count', 'usage_quality_status_20', 'usage_quality_status_21', 'payment_id', 'bill_month', 'amount_paid', 'payment_mode', 'payment_status', 'payment_quality_status']


In [121]:
payments_clean = payments_clean.drop("usage_quality_status")

In [122]:
gold_df = customers_clean \
    .join(plans_flat, "plan_id", "left") \
    .join(updated_usage_df, "customer_id", "left") \
    .join(payments_clean, "customer_id", "left")

In [126]:
from collections import Counter

duplicates = [c for c, cnt in Counter(gold_df.columns).items() if cnt > 1]
print(duplicates)

['usage_quality_status']


In [127]:
gold_df = gold_df.drop("usage_quality_status")

In [130]:
customers_clean = customers_clean.drop("usage_quality_status")
payments_clean = payments_clean.drop("usage_quality_status")

updated_usage_df = updated_usage_df.drop("usage_quality_status")

updated_usage_df = updated_usage_df.withColumnRenamed(
    "data_quality_status",
    "usage_quality_status"
)

In [131]:
gold_df = customers_clean \
    .join(plans_flat, "plan_id", "left") \
    .join(updated_usage_df, "customer_id", "left") \
    .join(payments_clean, "customer_id", "left")

In [132]:
gold_df.write.mode("overwrite") \
    .partitionBy("usage_month") \
    .parquet("gold/customer_usage_summary")

In [133]:
print(customers_clean.columns)
print(updated_usage_df.columns)
print(payments_clean.columns)

['customer_id', 'customer_name', 'city', 'state', 'age', 'gender', 'plan_id', 'status', 'customer_quality_status']
['usage_id', 'customer_id', 'usage_month', 'data_used_gb', 'call_minutes', 'sms_count']
['payment_id', 'customer_id', 'bill_month', 'amount_paid', 'payment_mode', 'payment_status', 'payment_quality_status']


In [134]:

before_count = usage_clean.count()

from pyspark.sql.functions import lit

usage_incremental_df = usage_incremental_df.withColumn(
    "data_quality_status",
    lit("Valid")
)
updated_usage_df = usage_clean.unionByName(
    usage_incremental_df,
    allowMissingColumns=True
)

updated_usage_df.write.mode("overwrite") \
.parquet("silver/usage_updated")

updated_usage_df = spark.read.parquet("silver/usage_updated")

after_count = updated_usage_df.count()

records_added = after_count - before_count

print("Before Incremental Load =", before_count)
print("After Incremental Load =", after_count)
print("Records Added =", records_added)

Before Incremental Load = 15
After Incremental Load = 20
Records Added = 5


In [135]:
customer_usage_summary = gold_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "call_minutes",
    "sms_count",
    "amount_paid",
    "payment_status"
)

customer_usage_summary.show(truncate=False)

+-----------+-------------+---------+------------+-------------------+------------+------------+---------+-----------+--------------+
|customer_id|customer_name|city     |plan_name   |usage_month        |data_used_gb|call_minutes|sms_count|amount_paid|payment_status|
+-----------+-------------+---------+------------+-------------------+------------+------------+---------+-----------+--------------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-03-01 00:00:00|55          |1100        |140      |499        |Success       |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-03-01 00:00:00|55          |1100        |140      |499        |Success       |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |1000        |130      |499        |Success       |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-02-01 00:00:00|50          |1000        |130      |499        |Success       |
|101        |Rahul Sharma |Hyderabad|Smart Basic |2026-01-01 0

In [136]:
from pyspark.sql.functions import *

plan_performance_report = gold_df.groupBy("plan_name") \
.agg(
    count("customer_id").alias("customer_count"),
    sum("amount_paid").alias("total_revenue"),
    avg("data_used_gb").alias("avg_data_usage")
)

plan_performance_report.show()

+------------+--------------+-------------+-----------------+
|   plan_name|customer_count|total_revenue|   avg_data_usage|
+------------+--------------+-------------+-----------------+
|        NULL|             2|            0|             NULL|
| Smart Basic|            13|         6487|53.69230769230769|
|Budget Saver|             2|          598|             11.0|
| Premium Max|             8|         9592|            61.25|
|  Smart Plus|             8|         6392|             33.5|
+------------+--------------+-------------+-----------------+



In [137]:
city_revenue_report = gold_df.groupBy("city") \
.agg(
    sum("amount_paid").alias("total_revenue")
)

city_revenue_report.show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         5293|
|    Kochi|         2398|
|  Chennai|         2994|
|   Mumbai|          299|
|     Pune|          799|
|    Delhi|         7993|
|Hyderabad|         3293|
+---------+-------------+



In [139]:
from pyspark.sql.functions import when, col

gold_df = gold_df.withColumn(
    "churn_risk",
    when(col("payment_status") == "Failed", "High Risk")
    .when(col("payment_status") == "Pending", "High Risk")
    .otherwise("Low Risk")
)

In [140]:
churn_risk_report = gold_df.filter(
    col("churn_risk") == "High Risk"
).select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "churn_risk"
)

churn_risk_report.show()

+-----------+-------------+---------+------------+--------------+----------+
|customer_id|customer_name|     city|   plan_name|payment_status|churn_risk|
+-----------+-------------+---------+------------+--------------+----------+
|        103|   Amit Kumar|   Mumbai|Budget Saver|        Failed| High Risk|
|        105|   Farhan Ali|    Delhi| Premium Max|       Pending| High Risk|
|        105|   Farhan Ali|    Delhi| Premium Max|       Pending| High Risk|
|        105|   Farhan Ali|    Delhi| Premium Max|       Pending| High Risk|
|        107|  Arjun Verma|Hyderabad|Budget Saver|       Pending| High Risk|
+-----------+-------------+---------+------------+--------------+----------+



In [142]:
from pyspark.sql.functions import *

gold_df = gold_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)

gold_df = gold_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes")
    .otherwise("No")
)

In [143]:
over_usage_report = gold_df.filter(
    col("over_usage_flag") == "Yes"
).select(
    "customer_id",
    "customer_name",
    "plan_name",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
)

over_usage_report.show()

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|  plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|        101| Rahul Sharma|Smart Basic|          55|           50|            5|
|        101| Rahul Sharma|Smart Basic|          55|           50|            5|
|        104|  Sneha Patel|Smart Basic|          62|           50|           12|
|        104|  Sneha Patel|Smart Basic|          62|           50|           12|
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
+-----------+-------------+-----------+------------+-------------+-------------+

